<a href="https://colab.research.google.com/github/Chanthul4054/E-Policing-An-Integrated-Spatio-Temporal-Crime-Forecasting-and-Decision-Support-System/blob/Spatio-Temporal-Crime-Prediction-System/Component_1_Model_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Selecting a Suitable Model

Model Benchmarking

Hyperparameter Tuning

Download/Export the Model

# Selecting a Suitable Model

#CRIME HOTSPOT PREDICTION - model predicts a single "Total Risk" score


#Most reliable metrices

### Import libraries

In [ ]:
import pandas as pd
import joblib
import json
import os
from google.colab import drive
import numpy as np

import time
from sklearn.metrics import (classification_report, roc_auc_score, precision_score,
                            recall_score, f1_score, confusion_matrix, make_scorer)

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold,  PredefinedSplit, GridSearchCV

from lightgbm import LGBMClassifier
import lightgbm as lgb

!pip install lightgbm
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import sklearn, lightgbm, joblib, pandas
print(f"scikit-learn=={sklearn.__version__}")
print(f"lightgbm=={lightgbm.__version__}")
print(f"pandas=={pandas.__version__}")
print(f"joblib=={joblib.__version__}")


scikit-learn==1.6.1
lightgbm==4.6.0
pandas==2.2.2
joblib==1.5.3


In [ ]:
"""
!pip install flask_cors
from flask_cors import CORS
"""

'\n!pip install flask_cors\nfrom flask_cors import CORS\n'

Load the training data

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

# Load Data(Trainibg and Validation)

import joblib
import numpy as np
from sklearn.model_selection import StratifiedKFold
import lightgbm as lgb

save_path = '/content/drive/MyDrive/DSGP/'

#Load correct file
bundle_path = os.path.join(save_path, 'all_crimes_data_FIXED.joblib')
all_data = joblib.load(bundle_path)

selected_crime = 'burglary'

# Extract with proper validation set
X_train = all_data[selected_crime]['X_train']
y_train = all_data[selected_crime]['y_train']
X_val = all_data[selected_crime]['X_val']
y_val = all_data[selected_crime]['y_val']


# Get imbalance ratio
imbalance_ratio = all_data[selected_crime]['imbalance_ratio']
scale_pos_weight = min(imbalance_ratio / 2, 20.0)

print(f"{selected_crime.upper()} Data Loaded!")
print(f"   Train: {len(X_train):,} samples")
print(f"   Val:   {len(X_val):,} samples")


#Load Test data with meta data
try:
    test_df_full = pd.read_parquet(save_path + 'test_df_with_metadata.parquet')
    has_metadata = True
except:
    print("test_df_with_metadata.parquet not found!")
    print("   Run preprocessing notebook first with the new save cell")
    has_metadata = False

if has_metadata:
    # Load feature order
    with open(save_path + 'feature_list.json', 'r') as f:
        feature_order = json.load(f)

    target_col = f'target_{selected_crime}_next_week'
    weeks = sorted(test_df_full['week_start'].unique())

print("ALL DATA LOADED SUCCESSFULLY")

✅ BURGLARY Data Loaded!
   Train: 13,916 samples
   Val:   2,982 samples
ALL DATA LOADED SUCCESSFULLY


####Applying SMOTE

SMOTE is applied in the training notebook to maintain flexibility—it allows experimenting with different SMOTE strategies per crime type without re-running expensive preprocessing, and ensures validation/test sets remain real-world distributions for accurate performance measurement.

In [ ]:

from imblearn.over_sampling import SMOTE
import numpy as np

print("="*70)
print("APPLYING SMOTE TO TRAINING DATA")
print("="*70)

# Get original statistics
n_pos_original = int(y_train.sum())
n_neg_original = int(len(y_train) - n_pos_original)

print(f"\nOriginal Training Data:")
print(f"   Positive (crimes):    {n_pos_original:,}")
print(f"   Negative (no crimes): {n_neg_original:,}")
print(f"   Total samples:        {len(y_train):,}")
print(f"   Imbalance ratio:      {imbalance_ratio:.1f}:1")
print(f"   Crime rate:           {(n_pos_original/len(y_train)*100):.2f}%")

# Determine SMOTE strategy
if imbalance_ratio > 50:
    smote_strategy = 0.20
    strategy_name = "Conservative (20%)"
elif imbalance_ratio > 20:
    smote_strategy = 0.25
    strategy_name = "Moderate (25%)"
else:
    smote_strategy = 0.30
    strategy_name = "Aggressive (30%)"

print(f"\n🔧 SMOTE Strategy: {strategy_name}")
print(f"   Target: Minority class = {smote_strategy*100:.0f}% of majority class")

# Apply SMOTE
print(f"\nApplying SMOTE...")
sm = SMOTE(sampling_strategy=smote_strategy, random_state=42)
X_train_smote, y_train_smote = sm.fit_resample(X_train, y_train)

# Calculate new statistics
n_pos_smote = int(y_train_smote.sum())
n_neg_smote = int(len(y_train_smote) - n_pos_smote)
new_ratio = n_neg_smote / n_pos_smote if n_pos_smote > 0 else 0

print(f"\nSMOTE Complete!")
print(f"\nAfter SMOTE:")
print(f"   Positive (crimes):    {n_pos_smote:,} (↑ {n_pos_smote - n_pos_original:,})")
print(f"   Negative (no crimes): {n_neg_smote:,} (no change)")
print(f"   Total samples:        {len(y_train_smote):,} (↑ {len(y_train_smote) - len(y_train):,})")
print(f"   New ratio:            {new_ratio:.1f}:1")
print(f"   New crime rate:       {(n_pos_smote/len(y_train_smote)*100):.2f}%")
print(f"   Synthetic crimes created: {n_pos_smote - n_pos_original:,}")

# Replace training data with SMOTE'd version
X_train = X_train_smote
y_train = y_train_smote

print(f"\nTraining data updated!")
print(f"   X_train shape: {X_train.shape}")
print(f"   y_train shape: {y_train.shape}")
print(f"   All subsequent training will use SMOTE'd data")

print("="*70)

APPLYING SMOTE TO TRAINING DATA

📊 Original Training Data:
   Positive (crimes):    308
   Negative (no crimes): 13,608
   Total samples:        13,916
   Imbalance ratio:      44.2:1
   Crime rate:           2.21%

🔧 SMOTE Strategy: Moderate (25%)
   Target: Minority class = 25% of majority class

⏳ Applying SMOTE...

✅ SMOTE Complete!

📊 After SMOTE:
   Positive (crimes):    3,402 (↑ 3,094)
   Negative (no crimes): 13,608 (no change)
   Total samples:        17,010 (↑ 3,094)
   New ratio:            4.0:1
   New crime rate:       20.00%
   Synthetic crimes created: 3,094

✅ Training data updated!
   X_train shape: (17010, 38)
   y_train shape: (17010,)
   All subsequent training will use SMOTE'd data


## LightGBM (Gradient Boosting Machine)

In [ ]:
#Initialize the model with Early Stopping capability
lgbm_model = lgb.LGBMClassifier(

    objective='binary',
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
    # Good defaults (no need for RandomSearch)
    learning_rate=0.01,
    n_estimators=1000,
    num_leaves=31,
    max_depth=10,
    min_child_samples=20,
    lambda_l1=1.0,
    lambda_l2=1.0,
    feature_fraction=0.8,
    subsample=0.8

)

In [ ]:
# Train on the RESAMPLED data with Early Stopping
lgbm_model.fit(
    X_train, y_train
)

LGBMClassifier(feature_fraction=0.8, lambda_l1=1.0, lambda_l2=1.0,
               learning_rate=0.01, max_depth=10, n_estimators=1000, n_jobs=-1,
               objective='binary', random_state=42, scale_pos_weight=20.0,
               subsample=0.8, verbosity=-1)

In [ ]:
import pandas as pd
import json
from sklearn.metrics import roc_auc_score
import numpy as np

print("="*70)
print("CORRECT EVALUATION: WEEKLY PERFORMANCE")
print("="*70)
print(f"\n📊 Test Period:")
print(f"   Weeks: {len(weeks)}")
print(f"   First: {weeks[0]}")
print(f"   Last:  {weeks[-1]}")

weekly_results = []
for week in weeks:
    week_data = test_df_full[test_df_full['week_start'] == week].copy()

    # Skip if less than 10 divisions (data quality issue)
    if len(week_data) < 10:
       continue

    # Get features and target
    X_week = week_data[feature_order].values
    y_week = week_data[target_col].values

    # Skip weeks with no crimes
    if y_week.sum() == 0:
        continue

    # Predict
    probs_week = lgbm_model.predict_proba(X_week)[:, 1]

    # Get top-10 from this week's divisions
    top10_idx = np.argsort(probs_week)[::-1][:10]
    crimes_caught = y_week[top10_idx].sum()
    total_crimes = y_week.sum()
    capture_rate = (crimes_caught / total_crimes) * 100

    weekly_results.append({
        'week': week,
        'n_divisions': len(week_data),
        'total_crimes': int(total_crimes),
        'crimes_caught': int(crimes_caught),
        'capture_rate': capture_rate
    })

# Calculate Summary statistics
results_df = pd.DataFrame(weekly_results)
avg_capture = results_df['capture_rate'].mean()
median_capture = results_df['capture_rate'].median()
random_baseline = (10 / 72) * 100  # 10 divisions out of 72
lift = avg_capture / random_baseline if random_baseline > 0 else 0

# Calculate Traditional ML Metrics
X_test_all = test_df_full[feature_order].values
y_test_all = test_df_full[target_col].values
y_test_proba_all = lgbm_model.predict_proba(X_test_all)[:, 1]
test_auc = roc_auc_score(y_test_all, y_test_proba_all)

# ============================================================================
# DISPLAY KEY METRICS - CLEAR & SEQUENTIAL
# ============================================================================

print("\n" + "="*70)
print("KEY PERFORMANCE METRICS")
print("="*70)

# Metric 1: Average Capture Rate
print(f"\nAVERAGE TOP-10 CAPTURE RATE")
print(f"    Value:          {avg_capture:.1f}%")
if avg_capture >= 40:
    status_avg = "EXCELLENT"
elif avg_capture >= 25:
    status_avg = "GOOD"
elif avg_capture >= random_baseline:
    status_avg = "FAIR"
else:
    status_avg = "POOR"


# Metric 2: Median Capture Rate
print(f"\nMEDIAN CAPTURE RATE")
print(f"    Value:          {median_capture:.1f}%")

diff = abs(avg_capture - median_capture)
if diff <= 5:
    status_median = "EXCELLENT (Highly Consistent)"
elif diff <= 10:
    status_median = "GOOD (Consistent)"
elif diff <= 20:
    status_median = "FAIR (Some Variance)"
else:
    status_median = "POOR (High Variance)"


# Metric 3: ROC-AUC
print(f"\nROC-AUC (Ranking Quality)")
print(f"    Value:          {test_auc:.4f}")

if test_auc >= 0.75:
    status_auc = "EXCELLENT"
elif test_auc >= 0.70:
    status_auc = "GOOD"
elif test_auc >= 0.65:
    status_auc = "FAIR"
else:
    status_auc = "POOR"



# Overall Summary
print("\n" + "="*70)
print("OVERALL ASSESSMENT")
print("="*70)

all_excellent = (avg_capture >= 40 and diff <= 10 and test_auc >= 0.75)
all_good = (avg_capture >= 25 and diff <= 20 and test_auc >= 0.70)

if all_excellent:
    overall = "EXCELLENT - READY FOR IMMEDIATE DEPLOYMENT"
elif all_good:
    overall = "GOOD - READY FOR DEPLOYMENT"
elif avg_capture >= random_baseline:
    overall = "FAIR - PILOT DEPLOYMENT RECOMMENDED"
else:
    overall = "POOR - NOT READY FOR DEPLOYMENT"

print(f"\n   {overall}")
print(f"\n   Lift over random:  {lift:.2f}x ({avg_capture:.1f}% vs {random_baseline:.1f}%)")
print(f"   Weeks evaluated:   {len(results_df)}")
print(f"   Consistency:       {'High' if diff <= 10 else 'Moderate' if diff <= 20 else 'Low'}")

# ============================================================================
# DETAILED BREAKDOWN (existing code)
# ============================================================================

print("\n" + "="*70)
print("DETAILED WEEKLY BREAKDOWN")
print("="*70)

print(f"\n📊 Weekly Statistics:")
print(f"   Weeks evaluated:        {len(results_df)}")
print(f"   Avg divisions per week: {results_df['n_divisions'].mean():.0f}")
print(f"   Best week:              {results_df['capture_rate'].max():.1f}%")
print(f"   Worst week:             {results_df['capture_rate'].min():.1f}%")

print(f"\n📋 Week-by-Week Performance:")
print(f"   {'Week':<12} {'Divisions':<10} {'Crimes':<8} {'Caught':<8} {'Rate':<10}")
print("   " + "-"*55)

for _, row in results_df.head(10).iterrows():
    week_str = str(row['week'])[:10]
    print(f"   {week_str:<12} {row['n_divisions']:<10} "
          f"{row['total_crimes']:<8} {row['crimes_caught']:<8} "
          f"{row['capture_rate']:.1f}%")

if len(results_df) > 10:
    print(f"   ... ({len(results_df) - 10} more weeks)")

print("\n" + "="*70)



CORRECT EVALUATION: WEEKLY PERFORMANCE

📊 Test Period:
   Weeks: 42
   First: 2024-06-24 00:00:00
   Last:  2025-09-15 00:00:00


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut


KEY PERFORMANCE METRICS

AVERAGE TOP-10 CAPTURE RATE
    Value:          42.7%

MEDIAN CAPTURE RATE
    Value:          41.7%

ROC-AUC (Ranking Quality)
    Value:          0.8033

OVERALL ASSESSMENT

   ✅ EXCELLENT - READY FOR IMMEDIATE DEPLOYMENT

   Lift over random:  3.07x (42.7% vs 13.9%)
   Weeks evaluated:   28
   Consistency:       High

DETAILED WEEKLY BREAKDOWN

📊 Weekly Statistics:
   Weeks evaluated:        28
   Avg divisions per week: 72
   Best week:              100.0%
   Worst week:             0.0%

📋 Week-by-Week Performance:
   Week         Divisions  Crimes   Caught   Rate      
   -------------------------------------------------------
   2024-07-01   72         1        1        100.0%
   2024-07-08   72         3        2        66.7%
   2024-07-15   73         1        0        0.0%
   2024-07-22   72         1        0        0.0%
   2024-08-05   72         2        0        0.0%
   2024-08-19   72         5        1        20.0%
   2024-08-26   72         1 

###Save model

In [ ]:
# Save it

model_filename = f'model_{selected_crime}.pkl'

joblib.dump(lgbm_model, model_filename)

print(f"Success! Optimized model for {selected_crime} saved as {model_filename}")

feature_names = feature_order
joblib.dump(feature_names, f'{selected_crime}_features.pkl')


✅ Success! Optimized model for burglary saved as model_burglary.pkl


['burglary_features.pkl']

#### Getting predictions


In [ ]:
import joblib
import json
from IPython.display import JSON, display


print("\n" + "="*80)
print("TESTING PREDICTIONS ON INFERENCE DATA")
print("="*80)

# 1. Load inference data
inference_df = pd.read_parquet('/content/drive/MyDrive/DSGP/inference_data_latest.parquet')


# 2. Remove duplicates
clean_inference = inference_df.drop_duplicates('gn_encoded', keep='last').copy()
print(f"   After dedup: {len(clean_inference)} unique GN divisions")

# 3. Load scaler and master features
scaler = joblib.load('/content/drive/MyDrive/DSGP/feature_scaler.joblib')
with open('/content/drive/MyDrive/DSGP/feature_list.json') as f:
    master_features = json.load(f)

# 4. Scale features - MATCH PREPROCESSING LOGIC (Exclude gn_encoded)
cols_to_scale = [col for col in master_features if col != 'gn_encoded']
df_for_scaler = clean_inference[cols_to_scale].fillna(0)
scaled_values = scaler.transform(df_for_scaler)

# Reconstruct DataFrame with scaled values
X_inference = pd.DataFrame(scaled_values, columns=cols_to_scale, index=clean_inference.index)
X_inference['gn_encoded'] = clean_inference['gn_encoded'] # Add back raw integer

# 5. Load model and feature order
model = joblib.load(f'model_{selected_crime}.pkl')
model_features = joblib.load(f'{selected_crime}_features.pkl')

# 6. Generate predictions using exact model feature order
X_final = X_inference[model_features]
risk_scores = model.predict_proba(X_final)[:, 1]

# 7. Map GN IDs to names
with open('/content/drive/MyDrive/DSGP/gn_name_mapping.json') as f:
    gn_map = {int(k): v for k, v in json.load(f).items()}

# 8. Construct results DataFrame
results = pd.DataFrame({
    'gn_encoded': clean_inference['gn_encoded'].values,
    'gn_name': [gn_map.get(int(gid), "Unknown") for gid in clean_inference['gn_encoded']],
    'risk_score': risk_scores
})

# 9. Format into final JSON structure
prediction_list = results.sort_values('risk_score', ascending=False).to_dict(orient='records')

final_output = {
    "crime_type": selected_crime,
    "predictions": prediction_list,
    "status": "success"
}

# 10. Display output in Colab
print("\n" + "="*80)
print(f"🚀 FINAL API-FORMATTED OUTPUT: {selected_crime.upper()}")
print("="*80)

display(JSON(final_output))


TESTING PREDICTIONS ON INFERENCE DATA

📊 Inference data loaded: (72, 46)
   After dedup: 72 unique GN divisions

🚀 FINAL API-FORMATTED OUTPUT: BURGLARY


<IPython.core.display.JSON object>

In [ ]:
from google.colab import files
files.download('model_burglary.pkl')
files.download('burglary_features.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>